# GIRP 调仓助手（日常使用，已对齐最新版）

这份 notebook 按你当前的最新版代码接口整理，和最新回测保持一致，核心对齐点包括：

- 使用 `intrinsic_backtest.py` 的最新接口；
- 参数块与回测 notebook 保持一致；
- 使用 **多周期 TSMOM 门控** 后，再计算 GIRP / IRP 目标权重；
- `activity_field` 默认使用 `vol`；
- `market` 会同时读取 `amount`（用于成交额约束）和当前活跃度字段；
- 当前持仓若不在最新 `WATCHLIST` 中，目标权重会自动视为 0，可给出减仓/清仓建议。

## 使用说明

1. 在参数区填写：
   - `WATCHLIST`
   - `CURRENT_LOTS`
   - `COST_PRICE`
   - `AVAILABLE_CASH`
2. 运行全部单元；
3. notebook 会：
   - 读取本地数据库最新数据；
   - 用**最新可见交易日收盘后**的信息生成信号；
   - 给出**下一个交易日执行**的参考调仓建议；
4. 由于下一个交易日的真实成交价未知，调仓建议中的成交价与手数是基于**最新可见价格口径**做的参考估算，实际成交时会有偏差。

## 与回测对齐的关键说明

- 回测中：`t` 日收盘后生成信号，`t+1` 日执行；
- 本 notebook 中：信号日期使用最新可见交易日，调仓建议面向下一个交易日；
- 由于 `t+1` 日价格未知，这里用最新可见的 `execution_price_type` 对应价格做参考测算。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_data import create_manager, today_str, load_tushare_token
from backtest import (
    build_market_matrices,
    get_execution_price_matrix,
    rebalance_to_target_weights,
    calc_actual_weights,
    calc_portfolio_value,
)
from intrinsic_backtest import (
    compute_intrinsic_target_weights_on_date,
    build_multi_period_gate_signal_matrix,
)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

# ========= 连接数据库 =========
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", Path(DB_PATH).resolve())

In [ ]:
# ========= 输入区：资产池 / 当前持仓 / 成本价 / 现金 / 参数 =========

WATCHLIST = [
    "510880.SH",  # 红利ETF (华泰柏瑞)
    "511090.SH",  # 30年国债ETF
    "518880.SH",  # 黄金ETF
    "161226.SZ",  # 白银LOF
    "501018.SH",  # 南方原油LOF
    "159985.SZ",  # 豆粕ETF
    "513100.SH",  # 纳斯达克100ETF (国泰)
]

# 当前持仓：按“手”填写，1 手 = 100 份（默认）
CURRENT_LOTS = {
    "510880.SH": 0,
    "511090.SH": 0,
    "518880.SH": 0,
    "161226.SZ": 0,
    "501018.SH": 0,
    "159985.SZ": 0,
    "513100.SH": 0,
}

# 成本价：按“每份/每股价格”填写；没有就写 0 或删掉
COST_PRICE = {
    "510880.SH": 0.0,
    "511090.SH": 0.0,
    "518880.SH": 0.0,
    "161226.SZ": 0.0,
    "501018.SH": 0.0,
    "159985.SZ": 0.0,
    "513100.SH": 0.0,
}

AVAILABLE_CASH = 30_000.0

# ========= 数据刷新与导出 =========
DO_REFRESH_RECENT = True
RECENT_LOOKBACK_DAYS = 7
END_DATE = today_str()

EXPORT_DIR = Path("data/exports_intrinsic_rebalance")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ========= 回测一致参数 =========
BACKTEST_PARAMS = {
    "initial_cash": 1_000_000.0,
    "lookback_window": 20,            # lbw：本征协方差矩阵的风险窗口
    "long_lookback_window": 252,      # llbw：zr / zv 的长期标准化窗口
    "activity_field": "vol",          # amount / vol
    "rebalance_freq": "M",            # D / W / M / Q / Y
    "execution_price_type": "avg",    # open / close / high / low / avg
    "fee_rate_buy": 0.0005,
    "fee_rate_sell": 0.0005,
    "lot_size": 100,
    "max_trade_amount_ratio": 0.05,
    "amount_unit_scale": 1000.0,      # Tushare 的 amount 常见口径为千元
    "use_drift_trigger": False,
    "drift_threshold": 0.05,
    "risk_free_rate": 0.0,
    "annualization": 252,
    "time_momentum_filter_window": 120,
    "cap_single_asset_to_full_pool": False,
}

TSMOM_GATE_PARAMS = {
    "lookback": [21, 63, 189],
    "skip_recent": [0, 0, 0],
    "signal_type": "sign",
    "use_excess_returns": True,
    "annual_rf": 0.02,
    "annualization": 252,
    "execution_lag": 0,
    "combination_method": "mean",
    "horizon_weights": None,
    "signal_clip": 3.0,
    "zero_to_nan": False,
    "gate_type": "directional",
    "gate_threshold": 0.34,
}

IRP_PREPARE_KWARGS = {
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,
}

ACTIVITY_PREPARE_KWARGS = {
    "ffill": False,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,
}

IRP_WEIGHT_KWARGS = {
    "annualization": 252,
    "drop_any_na": True,
    "long_only": True,
    "return_type": "log",
    "tol": 1e-12,
    "maxiter": 10000,
}

valuation_ffill_limit = 5

print("watchlist =", WATCHLIST)
print("current lots =", CURRENT_LOTS)
print("available cash =", AVAILABLE_CASH)
print("export_dir =", EXPORT_DIR.resolve())
print("params loaded")
display(pd.Series(BACKTEST_PARAMS, name="value").to_frame())
print("TSMOM gate params:")
display(pd.Series(TSMOM_GATE_PARAMS, name="value").to_frame())

In [ ]:
# ========= 刷新最近数据（可选） + 读取本地市场数据并构建 market =========
TRADE_UNIVERSE = sorted(set(WATCHLIST) | set(CURRENT_LOTS.keys()) | set(COST_PRICE.keys()))

if DO_REFRESH_RECENT:
    recent_summary = manager.refresh_recent_daily_prices(
        ts_codes=TRADE_UNIVERSE,
        lookback_days=RECENT_LOOKBACK_DAYS,
        end_date=END_DATE,
    )
    print("最近数据刷新结果：")
    display(pd.Series(recent_summary, name="value").to_frame())
else:
    print("已跳过最近数据刷新。")

gate_lookback = list(TSMOM_GATE_PARAMS.get("lookback", []))
gate_skip_recent = TSMOM_GATE_PARAMS.get("skip_recent", 0)
if np.isscalar(gate_skip_recent):
    gate_skip_recent = [int(gate_skip_recent)] * len(gate_lookback)
else:
    gate_skip_recent = [int(x) for x in gate_skip_recent]

gate_history_need = max([lb + sk + 1 for lb, sk in zip(gate_lookback, gate_skip_recent)], default=0)
ma_history_need = int(BACKTEST_PARAMS["time_momentum_filter_window"] or 0)
llbw_history_need = int(BACKTEST_PARAMS["long_lookback_window"]) + 1

required_history_bars = max(llbw_history_need, gate_history_need, ma_history_need, 2)
exchange = manager.config.default_exchange

open_trade_dates = pd.Index(manager.store.get_open_trade_dates(
    exchange=exchange,
    end_date=END_DATE,
)).astype(str)
open_trade_dates = open_trade_dates.sort_values()

if len(open_trade_dates) == 0:
    raise ValueError("数据库交易日历为空，请先更新 trade_calendar。")

preload_idx = max(0, len(open_trade_dates) - required_history_bars)
data_start_date = str(open_trade_dates[preload_idx])

raw_prices = manager.store.get_daily_prices(
    ts_codes=TRADE_UNIVERSE,
    start_date=data_start_date,
    end_date=END_DATE,
)

if raw_prices.empty:
    raise ValueError("raw_prices 为空，请检查数据库、资产池或日期区间。")

fields = ["open", "high", "low", "close", "amount"]
if BACKTEST_PARAMS["activity_field"] not in fields:
    fields.append(BACKTEST_PARAMS["activity_field"])

market = build_market_matrices(
    data=raw_prices,
    fields=tuple(fields),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

close_px = market["close"].copy()
val_px = close_px.ffill(limit=valuation_ffill_limit)
exec_px = get_execution_price_matrix(
    market,
    execution_price_type=BACKTEST_PARAMS["execution_price_type"],
)
amount_px = market["amount"].copy()

latest_signal_date = close_px.index.max()
val_today = val_px.loc[latest_signal_date]
exec_today = exec_px.loc[latest_signal_date]
amount_today = amount_px.loc[latest_signal_date]

# 为避免反复运行 notebook 时改写原字典，这里单独构造局部 kwargs
irp_prepare_kwargs = dict(IRP_PREPARE_KWARGS)
irp_prepare_kwargs["calendar"] = market["close"].index

activity_prepare_kwargs = dict(ACTIVITY_PREPARE_KWARGS)
activity_prepare_kwargs["calendar"] = market["close"].index

coverage = raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"]).sort_index()

print("END_DATE =", END_DATE)
print("data_start_date =", data_start_date)
print("required_history_bars =", required_history_bars)
print("latest_signal_date =", latest_signal_date)
print("raw_prices shape =", raw_prices.shape)
display(raw_prices.head())
print("资产覆盖情况：")
display(coverage)

for k, v in market.items():
    print(k, v.shape)

In [ ]:
# ========= 计算当前组合状态 =========
codes = close_px.columns.tolist()

current_shares = pd.Series(0, index=codes, dtype=int)
for code, lots in CURRENT_LOTS.items():
    if code in current_shares.index:
        current_shares.loc[code] = int(lots) * int(BACKTEST_PARAMS["lot_size"])

cost_price_series = pd.Series(0.0, index=codes, dtype=float)
for code, px in COST_PRICE.items():
    if code in cost_price_series.index:
        cost_price_series.loc[code] = float(px)

current_market_value = (current_shares * val_today.reindex(codes)).fillna(0.0)
current_actual_weights = calc_actual_weights(
    current_shares,
    val_today,
    cash=AVAILABLE_CASH,
).reindex(codes).fillna(0.0)
portfolio_value = calc_portfolio_value(current_shares, val_today, AVAILABLE_CASH)
cash_weight = AVAILABLE_CASH / portfolio_value if portfolio_value > 0 else np.nan

current_summary = pd.DataFrame({
    "lots": current_shares / BACKTEST_PARAMS["lot_size"],
    "shares": current_shares,
    "cost_price": cost_price_series,
    "valuation_price": val_today.reindex(codes),
    "reference_execution_price": exec_today.reindex(codes),
    "market_value": current_market_value,
    "actual_weight": current_actual_weights,
})

current_summary["pnl"] = (
    current_summary["valuation_price"] - current_summary["cost_price"]
) * current_summary["shares"]

current_summary["pnl_pct"] = np.where(
    current_summary["cost_price"] > 0,
    current_summary["valuation_price"] / current_summary["cost_price"] - 1.0,
    np.nan,
)

print("current portfolio value =", round(portfolio_value, 2))
print("cash =", round(AVAILABLE_CASH, 2))
print("cash weight =", round(cash_weight, 4))
display(current_summary)

In [ ]:
# ========= 计算最新多周期门控（与回测一致） =========
watchlist_market = {k: v.reindex(columns=WATCHLIST) for k, v in market.items()}

gate_signal_df = build_multi_period_gate_signal_matrix(
    market=watchlist_market,
    gate_price_field="close",
    multi_period_gate_params=TSMOM_GATE_PARAMS,
).reindex(index=watchlist_market["close"].index, columns=WATCHLIST).fillna(0.0)

gate_signal_today = gate_signal_df.loc[latest_signal_date].reindex(WATCHLIST).fillna(0.0)
gate_eligible_assets = gate_signal_today[gate_signal_today > 0].index.tolist()

gate_summary = pd.DataFrame({
    "gate_signal": gate_signal_today,
    "eligible": (gate_signal_today > 0).astype(int),
}, index=WATCHLIST)

print("latest gate signal on", latest_signal_date.strftime("%Y-%m-%d"))
print("eligible asset count =", int((gate_signal_today > 0).sum()))
display(gate_summary.sort_values(["eligible", "gate_signal"], ascending=[False, False]))

In [ ]:
# ========= 计算目标 GIRP / IRP 权重（先门控，再算权重） =========
target_weights_watchlist = compute_intrinsic_target_weights_on_date(
    market=watchlist_market,
    signal_date=latest_signal_date,
    lookback_window=BACKTEST_PARAMS["lookback_window"],
    long_lookback_window=BACKTEST_PARAMS["long_lookback_window"],
    activity_field=BACKTEST_PARAMS["activity_field"],
    irp_prepare_kwargs=irp_prepare_kwargs,
    activity_prepare_kwargs=activity_prepare_kwargs,
    irp_weight_kwargs=IRP_WEIGHT_KWARGS,
    time_momentum_filter_window=BACKTEST_PARAMS["time_momentum_filter_window"],
    eligible_assets=gate_eligible_assets,
    cap_single_asset_to_full_pool=BACKTEST_PARAMS["cap_single_asset_to_full_pool"],
).reindex(WATCHLIST).fillna(0.0)

if float(target_weights_watchlist.sum()) <= 0:
    print("本次门控后目标权重全为 0，组合将保持现金或仅处理非资产池旧持仓。")

# 扩展到全部交易范围：不在资产池的当前持仓，目标权重=0
target_weights = pd.Series(0.0, index=codes, dtype=float)
target_weights.loc[target_weights_watchlist.index] = target_weights_watchlist.values

target_weight_df = pd.DataFrame({
    "target_weight": target_weights.reindex(codes).fillna(0.0),
    "in_watchlist": pd.Index(codes).isin(WATCHLIST).astype(int),
    "gate_signal": gate_signal_today.reindex(codes).fillna(0.0),
})

display(target_weight_df.sort_values("target_weight", ascending=False))

In [ ]:
# ========= 生成调仓建议（下一个交易日参考） =========
new_shares, new_cash, trades_df, after_weights = rebalance_to_target_weights(
    current_shares=current_shares,
    cash=AVAILABLE_CASH,
    target_weights=target_weights,
    exec_prices=exec_today,
    val_prices=val_today,
    amount_series=amount_today,
    fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
    fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
    lot_size=BACKTEST_PARAMS["lot_size"],
    max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
    amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
    trade_date=latest_signal_date,
    allow_cash_residual=BACKTEST_PARAMS["cap_single_asset_to_full_pool"],
)

after_market_value = (new_shares * val_today.reindex(codes)).fillna(0.0)
after_weights = after_weights.reindex(codes).fillna(0.0)

if len(trades_df) > 0:
    buy_df = trades_df[trades_df["side"] == "BUY"].groupby("ts_code")["shares"].sum().rename("buy_shares")
    sell_df = trades_df[trades_df["side"] == "SELL"].groupby("ts_code")["shares"].sum().rename("sell_shares")
    trade_value_df = trades_df.groupby("ts_code")["trade_value"].sum().rename("trade_value")
    trade_cost_df = trades_df.groupby("ts_code")["cost"].sum().rename("trade_cost")
else:
    buy_df = pd.Series(dtype=float, name="buy_shares")
    sell_df = pd.Series(dtype=float, name="sell_shares")
    trade_value_df = pd.Series(dtype=float, name="trade_value")
    trade_cost_df = pd.Series(dtype=float, name="trade_cost")

rebalance_table = pd.DataFrame(index=codes)
rebalance_table["current_lots"] = current_shares / BACKTEST_PARAMS["lot_size"]
rebalance_table["current_shares"] = current_shares
rebalance_table["target_weight"] = target_weights.reindex(codes).fillna(0.0)
rebalance_table["current_weight"] = current_actual_weights.reindex(codes).fillna(0.0)
rebalance_table["after_weight"] = after_weights.reindex(codes).fillna(0.0)
rebalance_table["gate_signal"] = gate_signal_today.reindex(codes).fillna(0.0)
rebalance_table["eligible"] = (rebalance_table["gate_signal"] > 0).astype(int)
rebalance_table["valuation_price"] = val_today.reindex(codes)
rebalance_table["reference_execution_price"] = exec_today.reindex(codes)
rebalance_table = rebalance_table.join(buy_df, how="left").join(sell_df, how="left")
rebalance_table = rebalance_table.join(trade_value_df, how="left").join(trade_cost_df, how="left")
rebalance_table[["buy_shares", "sell_shares", "trade_value", "trade_cost"]] = rebalance_table[
    ["buy_shares", "sell_shares", "trade_value", "trade_cost"]
].fillna(0.0)

rebalance_table["net_trade_shares"] = rebalance_table["buy_shares"] - rebalance_table["sell_shares"]
rebalance_table["net_trade_lots"] = rebalance_table["net_trade_shares"] / BACKTEST_PARAMS["lot_size"]
rebalance_table["after_shares"] = new_shares.reindex(codes).fillna(0).astype(int)
rebalance_table["after_lots"] = rebalance_table["after_shares"] / BACKTEST_PARAMS["lot_size"]
rebalance_table["after_market_value"] = after_market_value

def classify_action(x):
    if x > 0:
        return "BUY"
    if x < 0:
        return "SELL"
    return "HOLD"

rebalance_table["suggestion"] = rebalance_table["net_trade_shares"].apply(classify_action)

print("cash before =", round(AVAILABLE_CASH, 2))
print("cash after  =", round(new_cash, 2))
print("cash delta  =", round(new_cash - AVAILABLE_CASH, 2))
print("trade count =", len(trades_df))
print("说明：手数与现金变化基于最新可见价格做参考估算，实际下个交易日执行会有偏差。")

display(rebalance_table.sort_values(["suggestion", "target_weight"], ascending=[True, False]))

In [ ]:
# ========= 仅看需要操作的标的 =========
action_table = rebalance_table[rebalance_table["suggestion"] != "HOLD"].copy()
display(action_table.sort_values(["suggestion", "net_trade_shares"], ascending=[True, False]))

print("交易明细：")
display(trades_df)

In [ ]:
# ========= 图形查看：当前权重 / 目标权重 / 调后权重 =========
compare_weights = pd.DataFrame({
    "current_weight": current_actual_weights.reindex(codes).fillna(0.0),
    "target_weight": target_weights.reindex(codes).fillna(0.0),
    "after_weight": after_weights.reindex(codes).fillna(0.0),
}).fillna(0.0)

display(compare_weights.sort_values("target_weight", ascending=False))

compare_weights.plot(kind="bar", figsize=(14, 5))
plt.title("Current vs Target vs After-Rebalance Weights (GIRP/IRP)")
plt.ylabel("Weight")
plt.tight_layout()
plt.show()

In [ ]:
# ========= 导出建议 =========
current_summary.to_csv(EXPORT_DIR / "girp_rebalance_current_summary.csv")
gate_summary.to_csv(EXPORT_DIR / "girp_rebalance_gate_summary.csv")
target_weight_df.to_csv(EXPORT_DIR / "girp_rebalance_target_weights.csv")
rebalance_table.to_csv(EXPORT_DIR / "girp_rebalance_suggestion.csv")
trades_df.to_csv(EXPORT_DIR / "girp_rebalance_trades_detail.csv", index=False)

run_meta = pd.Series({
    "signal_date": pd.Timestamp(latest_signal_date).strftime("%Y%m%d"),
    "end_date": END_DATE,
    "data_start_date": data_start_date,
    "activity_field": BACKTEST_PARAMS["activity_field"],
    "lookback_window": BACKTEST_PARAMS["lookback_window"],
    "long_lookback_window": BACKTEST_PARAMS["long_lookback_window"],
    "execution_price_type": BACKTEST_PARAMS["execution_price_type"],
    "time_momentum_filter_window": BACKTEST_PARAMS["time_momentum_filter_window"],
    "gate_lookback": TSMOM_GATE_PARAMS["lookback"],
    "gate_threshold": TSMOM_GATE_PARAMS["gate_threshold"],
    "eligible_asset_count": int((gate_signal_today > 0).sum()),
    "target_weight_sum": float(target_weights.sum()),
})
run_meta.to_csv(EXPORT_DIR / "girp_rebalance_run_meta.csv")

print("export done")
print("export_dir =", EXPORT_DIR.resolve())
display(run_meta.to_frame("value"))

In [ ]:
# ========= 精简版调仓建议（含标的名） =========
signal_date = pd.Timestamp(latest_signal_date)
signal_date_str = signal_date.strftime("%Y-%m-%d")

# 读取标的名称映射
inst_df = manager.store.get_instruments(listed_only=False)
name_map = dict(zip(inst_df["ts_code"], inst_df["name"]))

# 尝试从交易日历里找下一个交易日
next_trade_date = None
try:
    cal_df = manager.store.get_trade_calendar(
        exchange=manager.config.default_exchange,
        start_date=signal_date.strftime("%Y%m%d"),
        end_date="20300101",
        is_open=1,
    )
    future_open_dates = pd.to_datetime(cal_df["cal_date"], format="%Y%m%d", errors="coerce")
    future_open_dates = future_open_dates[future_open_dates > signal_date]
    if len(future_open_dates) > 0:
        next_trade_date = future_open_dates.iloc[0]
except Exception:
    next_trade_date = None

simple_advice = rebalance_table[[
    "current_lots",
    "after_lots",
    "target_weight",
    "gate_signal",
    "suggestion",
]].copy()

simple_advice = simple_advice.reset_index().rename(columns={
    "index": "ts_code",
    "current_lots": "调整前手数",
    "after_lots": "调整后手数",
    "target_weight": "目标权重",
    "gate_signal": "门控信号",
    "suggestion": "建议动作",
})

simple_advice["name"] = simple_advice["ts_code"].map(name_map)
simple_advice = simple_advice[[
    "ts_code", "name", "调整前手数", "调整后手数", "目标权重", "门控信号", "建议动作"
]]

simple_advice["调整前手数"] = simple_advice["调整前手数"].astype(int)
simple_advice["调整后手数"] = simple_advice["调整后手数"].astype(int)

print(f"当前读取到的最新信号日期：{signal_date_str}")
print(
    f"GIRP 参数：lbw={BACKTEST_PARAMS['lookback_window']}, "
    f"llbw={BACKTEST_PARAMS['long_lookback_window']}, "
    f"activity={BACKTEST_PARAMS['activity_field']}"
)
print(
    f"门控参数：lookback={TSMOM_GATE_PARAMS['lookback']}, "
    f"threshold={TSMOM_GATE_PARAMS['gate_threshold']}"
)

if next_trade_date is not None:
    print(f"建议于下一个交易日（{next_trade_date.strftime('%Y-%m-%d')}）参考执行如下调整：")
else:
    print("建议于下一个交易日参考执行如下调整：")

display(simple_advice)

In [ ]:

# ========= 完整调仓报告（打印 + Markdown 落盘） =========
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown

STRATEGY_NAME = "GIRP"
REPORT_ROOT = Path("Trading_advice")
REPORT_FORMAT = "md"   # 当前默认输出 md
SHOW_ONLY_ACTIONS_IN_MAIN_TABLE = True

required_vars = [
    "manager", "WATCHLIST", "BACKTEST_PARAMS", "TSMOM_GATE_PARAMS",
    "latest_signal_date", "data_start_date", "END_DATE",
    "current_summary", "gate_summary", "target_weight_df",
    "rebalance_table", "trades_df", "AVAILABLE_CASH", "new_cash",
    "portfolio_value", "target_weights", "current_actual_weights",
]
missing_vars = [x for x in required_vars if x not in globals()]
if missing_vars:
    raise NameError(f"以下变量不存在，请先运行前面的 cells：{missing_vars}")

signal_date = pd.Timestamp(latest_signal_date)
signal_date_str = signal_date.strftime("%Y-%m-%d")
signal_date_compact = signal_date.strftime("%Y%m%d")

# 尝试从交易日历读取下一个交易日（仅日期提示，不使用 next open 估算）
next_trade_date = None
try:
    cal_df = manager.store.get_trade_calendar(
        exchange=manager.config.default_exchange,
        start_date=signal_date.strftime("%Y%m%d"),
        end_date="20300101",
        is_open=1,
    )
    future_open_dates = pd.to_datetime(cal_df["cal_date"], format="%Y%m%d", errors="coerce")
    future_open_dates = future_open_dates[future_open_dates > signal_date]
    if len(future_open_dates) > 0:
        next_trade_date = future_open_dates.iloc[0]
except Exception:
    next_trade_date = None

inst_df = manager.store.get_instruments(listed_only=False)
name_map = dict(zip(inst_df["ts_code"], inst_df["name"]))

report_dir = REPORT_ROOT / STRATEGY_NAME
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f"{signal_date_compact}.md"

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def _fmt_num(x, digits=4, pct=False):
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if pct:
        return f"{x:.{digits}%}"
    return f"{x:,.{digits}f}"

def _fmt_int_like(x):
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return ""
    try:
        return str(int(round(float(x))))
    except Exception:
        return str(x)

def _md_escape(x):
    s = "" if x is None else str(x)
    return s.replace("|", r"\|").replace("\n", "<br>")

def df_to_md(df):
    if df is None or len(df) == 0:
        return "_无数据_"
    x = df.copy()
    x = x.replace({np.nan: ""})
    headers = list(x.columns)
    lines = []
    lines.append("| " + " | ".join(_md_escape(h) for h in headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in x.iterrows():
        lines.append("| " + " | ".join(_md_escape(v) for v in row.tolist()) + " |")
    return "\n".join(lines)

# ========== 构造状态解释 ==========
report_table = rebalance_table.copy()
report_table = report_table.reset_index().rename(columns={"index": "ts_code"})
report_table["name"] = report_table["ts_code"].map(name_map)
report_table["in_watchlist"] = report_table["ts_code"].isin(WATCHLIST).astype(int)
report_table["current_position_flag"] = (report_table["current_shares"].fillna(0) > 0).astype(int)
report_table["target_position_flag"] = (report_table["target_weight"].fillna(0) > 0).astype(int)

def explain_row(row):
    reasons = []

    in_watchlist = int(row.get("in_watchlist", 0))
    gate_signal = _safe_float(row.get("gate_signal", 0.0))
    current_shares = _safe_float(row.get("current_shares", 0.0))
    target_weight = _safe_float(row.get("target_weight", 0.0))
    net_trade_shares = _safe_float(row.get("net_trade_shares", 0.0))

    has_position = current_shares > 0
    target_has_position = target_weight > 0

    # 先解释“为什么在/不在目标组合”
    if in_watchlist == 0:
        reasons.append("旧持仓/不在当前资产池")
    elif gate_signal <= 0:
        reasons.append("门控未通过")
    elif gate_signal > 0 and not target_has_position:
        reasons.append("通过门控但目标权重为0")
    elif target_has_position:
        reasons.append("进入目标组合")

    # 再解释“实际调仓建议”
    if net_trade_shares > 0:
        reasons.append("建议买入")
    elif net_trade_shares < 0:
        reasons.append("建议卖出")
    else:
        # 没有净交易时，细分状态，避免笼统写“建议保持”
        if has_position and target_has_position:
            reasons.append("建议保持持仓")
        elif (not has_position) and (not target_has_position):
            reasons.append("建议保持空仓")
        elif has_position and (not target_has_position):
            reasons.append("目标应清仓，但本次估算未形成卖单")
        elif (not has_position) and target_has_position:
            reasons.append("目标应建仓，但本次估算未形成买单")
        else:
            reasons.append("建议保持")

    return "；".join(reasons)

report_table["status_explanation"] = report_table.apply(explain_row, axis=1)

# ========== 成交额约束快照 ==========
amount_scale = float(BACKTEST_PARAMS.get("amount_unit_scale", 1.0))
max_trade_amount_ratio = BACKTEST_PARAMS.get("max_trade_amount_ratio", None)
if "amount_today" in globals() and amount_today is not None:
    amount_today_series = pd.Series(amount_today).reindex(report_table["ts_code"]).astype(float)
else:
    amount_today_series = pd.Series(np.nan, index=report_table["ts_code"])

report_table["amount_today"] = amount_today_series.values
if max_trade_amount_ratio is None:
    report_table["trade_value_cap"] = np.nan
else:
    report_table["trade_value_cap"] = report_table["amount_today"] * amount_scale * float(max_trade_amount_ratio)

report_table["reference_trade_value"] = (
    report_table["reference_execution_price"].astype(float).fillna(0.0)
    * report_table["net_trade_shares"].abs().astype(float).fillna(0.0)
)
report_table["cap_usage_ratio"] = np.where(
    (report_table["trade_value_cap"].astype(float) > 0) & np.isfinite(report_table["trade_value_cap"].astype(float)),
    report_table["reference_trade_value"] / report_table["trade_value_cap"],
    np.nan,
)

# ========== 组合摘要 ==========
eligible_count = int((gate_summary["eligible"] > 0).sum()) if "eligible" in gate_summary.columns else int((gate_summary.iloc[:, 0] > 0).sum())
target_weight_sum = float(pd.Series(target_weights).fillna(0.0).sum())
trade_count = int(len(trades_df))
action_count = int((report_table["suggestion"] != "HOLD").sum())
portfolio_value_after_ref = float(((report_table["after_shares"].astype(float) * report_table["valuation_price"].astype(float)).fillna(0.0)).sum() + float(new_cash))
cash_before_weight = float(AVAILABLE_CASH) / float(portfolio_value) if float(portfolio_value) > 0 else np.nan
cash_after_weight_ref = float(new_cash) / float(portfolio_value_after_ref) if float(portfolio_value_after_ref) > 0 else np.nan

# ========== 对外展示表 ==========
main_cols = [
    "ts_code", "name", "suggestion", "status_explanation",
    "current_lots", "after_lots", "net_trade_lots",
    "current_weight", "target_weight", "after_weight",
    "gate_signal", "eligible",
    "valuation_price", "reference_execution_price",
    "reference_trade_value", "trade_value_cap", "cap_usage_ratio",
    "trade_cost",
]

main_table = report_table[main_cols].copy()
if SHOW_ONLY_ACTIONS_IN_MAIN_TABLE:
    shown_main_table = main_table[main_table["suggestion"] != "HOLD"].copy()
    if len(shown_main_table) == 0:
        shown_main_table = main_table.copy()
else:
    shown_main_table = main_table.copy()

for col in ["current_lots", "after_lots", "net_trade_lots"]:
    shown_main_table[col] = shown_main_table[col].map(lambda x: _fmt_num(x, digits=2))
for col in ["current_weight", "target_weight", "after_weight", "cap_usage_ratio"]:
    shown_main_table[col] = shown_main_table[col].map(lambda x: _fmt_num(x, digits=4, pct=(col != "cap_usage_ratio")))
shown_main_table["cap_usage_ratio"] = shown_main_table["cap_usage_ratio"].replace("", np.nan).map(lambda x: _fmt_num(x, digits=2, pct=True) if pd.notna(x) else "")
for col in ["gate_signal", "valuation_price", "reference_execution_price"]:
    shown_main_table[col] = shown_main_table[col].map(lambda x: _fmt_num(x, digits=4))
for col in ["reference_trade_value", "trade_value_cap", "trade_cost"]:
    shown_main_table[col] = shown_main_table[col].map(lambda x: _fmt_num(x, digits=2))
shown_main_table["eligible"] = shown_main_table["eligible"].map(_fmt_int_like)

position_cols = [
    "ts_code", "name", "current_lots", "current_weight", "target_weight", "after_lots", "after_weight",
    "valuation_price", "reference_execution_price", "status_explanation"
]
position_table = report_table[
    (report_table["current_shares"].fillna(0) != 0) | (report_table["target_weight"].fillna(0) > 0)
][position_cols].copy()
for col in ["current_lots", "after_lots"]:
    position_table[col] = position_table[col].map(lambda x: _fmt_num(x, digits=2))
for col in ["current_weight", "target_weight", "after_weight"]:
    position_table[col] = position_table[col].map(lambda x: _fmt_num(x, digits=4, pct=True))
for col in ["valuation_price", "reference_execution_price"]:
    position_table[col] = position_table[col].map(lambda x: _fmt_num(x, digits=4))

gate_report = gate_summary.copy().reset_index().rename(columns={"index": "ts_code"})
gate_report["name"] = gate_report["ts_code"].map(name_map)
gate_report = gate_report[["ts_code", "name", "gate_signal", "eligible"]]
gate_report["gate_signal"] = gate_report["gate_signal"].map(lambda x: _fmt_num(x, digits=4))
gate_report["eligible"] = gate_report["eligible"].map(_fmt_int_like)

trades_report = trades_df.copy()
if len(trades_report) > 0:
    trades_report["ts_code"] = trades_report["ts_code"].astype(str)
    trades_report["name"] = trades_report["ts_code"].map(name_map)
    trades_report["trade_date"] = pd.to_datetime(trades_report["trade_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    trades_report["price"] = trades_report["price"].map(lambda x: _fmt_num(x, digits=4))
    trades_report["shares"] = trades_report["shares"].map(_fmt_int_like)
    trades_report["trade_value"] = trades_report["trade_value"].map(lambda x: _fmt_num(x, digits=2))
    trades_report["cost"] = trades_report["cost"].map(lambda x: _fmt_num(x, digits=2))
    trades_report = trades_report[["trade_date", "ts_code", "name", "side", "price", "shares", "trade_value", "cost"]]

# ========== Markdown 正文 ==========
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

report_lines = []
report_lines.append(f"# {STRATEGY_NAME} 调仓建议报告")
report_lines.append("")
report_lines.append("## 基本信息")
report_lines.append("")
report_lines.append(f"- 策略名：`{STRATEGY_NAME}`")
report_lines.append(f"- 信号日：`{signal_date_str}`")
report_lines.append(f"- 信号使用数据截止：`{signal_date_str}` 收盘后")
if next_trade_date is not None:
    report_lines.append(f"- 计划执行日：`{pd.Timestamp(next_trade_date).strftime('%Y-%m-%d')}`（仅交易日提示）")
else:
    report_lines.append("- 计划执行日：下一个交易日（交易日历未成功解析具体日期）")
report_lines.append(f"- 数据起始：`{data_start_date}`")
report_lines.append(f"- 回测/助手参数截止日：`{END_DATE}`")
report_lines.append(f"- 报告生成时间：`{generated_at}`")
report_lines.append("")
report_lines.append("> 说明：本报告使用 **截止信号日收盘后可见的数据** 生成调仓建议。由于下一个交易日的真实成交价未知，下面的手数、成交额、调后现金与调后权重均基于 **最新可见参考执行价** 的估算，不使用 next open 预估。")
report_lines.append("")
report_lines.append("## 参数快照")
report_lines.append("")
report_lines.append(f"- GIRP：`lbw={BACKTEST_PARAMS['lookback_window']}`，`llbw={BACKTEST_PARAMS['long_lookback_window']}`，`activity_field={BACKTEST_PARAMS['activity_field']}`")
report_lines.append(f"- 调仓：`rebalance_freq={BACKTEST_PARAMS['rebalance_freq']}`，`execution_price_type={BACKTEST_PARAMS['execution_price_type']}`，`lot_size={BACKTEST_PARAMS['lot_size']}`")
report_lines.append(f"- 成本：`buy_fee={BACKTEST_PARAMS['fee_rate_buy']}`，`sell_fee={BACKTEST_PARAMS['fee_rate_sell']}`")
report_lines.append(f"- 成交额约束：`max_trade_amount_ratio={BACKTEST_PARAMS['max_trade_amount_ratio']}`，`amount_unit_scale={BACKTEST_PARAMS['amount_unit_scale']}`")
report_lines.append(f"- 多周期门控：`lookback={TSMOM_GATE_PARAMS['lookback']}`，`threshold={TSMOM_GATE_PARAMS['gate_threshold']}`，`combination_method={TSMOM_GATE_PARAMS['combination_method']}`")
report_lines.append("")
report_lines.append("## 核心摘要")
report_lines.append("")
report_lines.append(f"- 当前组合估值：`{_fmt_num(portfolio_value, digits=2)}`")
report_lines.append(f"- 当前现金：`{_fmt_num(AVAILABLE_CASH, digits=2)}`，当前现金权重：`{_fmt_num(cash_before_weight, digits=2, pct=True)}`")
report_lines.append(f"- 参考调仓后现金：`{_fmt_num(new_cash, digits=2)}`，参考调仓后现金权重：`{_fmt_num(cash_after_weight_ref, digits=2, pct=True)}`")
report_lines.append(f"- 门控通过资产数：`{eligible_count}` / `{len(WATCHLIST)}`")
report_lines.append(f"- 目标风险资产权重和：`{_fmt_num(target_weight_sum, digits=2, pct=True)}`")
report_lines.append(f"- 建议操作资产数：`{action_count}`")
report_lines.append(f"- 参考成交笔数：`{trade_count}`")
report_lines.append("")
report_lines.append("## 调仓建议主表")
report_lines.append("")
report_lines.append(df_to_md(shown_main_table))
report_lines.append("")
report_lines.append("## 当前持仓与目标组合概览")
report_lines.append("")
report_lines.append(df_to_md(position_table))
report_lines.append("")
report_lines.append("## 门控结果")
report_lines.append("")
report_lines.append(df_to_md(gate_report))
report_lines.append("")
report_lines.append("## 参考成交明细")
report_lines.append("")
report_lines.append(df_to_md(trades_report))
report_lines.append("")
report_lines.append("## 备注")
report_lines.append("")
report_lines.append("- `status_explanation` 用于解释该资产为何进入或退出本次组合。")
report_lines.append("- `reference_execution_price` 是基于最新可见价格口径生成的参考执行价，只用于估算，不代表下个交易日真实成交价。")
report_lines.append("- `trade_value_cap` 为按当日成交额上限推导出的单日允许成交额；`cap_usage_ratio` 越接近或超过 100%，越需要注意实盘可执行性。")
report_lines.append("- 若某旧持仓不在当前 `WATCHLIST` 中，则本报告会把它视为目标权重 0，并在建议中体现减仓/清仓。")
report_text = "\n".join(report_lines)

# Notebook 内展示
display(Markdown(report_text[:30000] if len(report_text) > 30000 else report_text))

# 写入磁盘
report_path.write_text(report_text, encoding="utf-8")

print(f"完整调仓报告已保存：{report_path.resolve()}")
print("主表展示行数 =", len(shown_main_table))
